# DREAM AI — first independent model boot (Kaggle GPU)

This notebook boots DREAM on the openly licensed `Qwen/Qwen3-1.7B` pretrained weights. It does **not** train a model from random initialization and does not call a hosted AI inference API. Kaggle downloads the model weights and runs inference on its assigned GPU.

**Purpose of this first run:** prove that DREAM's starting brain loads and answers a test prompt on Kaggle. This is a bootstrap/inference test, not yet continuous training or a permanent server. Kaggle sessions are temporary; the model must later be hosted on a persistent GPU/CPU machine for always-available chat.

Before running: set Notebook accelerator to **GPU** and turn **Internet on** in Kaggle notebook settings. No Kaggle Dataset attachment or `approved-records.jsonl` is required for this bootstrap test.


In [ ]:
import sys, subprocess, importlib.util
packages = ['transformers>=4.51.3', 'accelerate>=1.0.0', 'safetensors']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Runtime packages installed.')


In [ ]:
import torch, json, os, platform
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen3-1.7B'
OUTPUT_DIR = Path('/kaggle/working/dream-qwen3-bootstrap')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Kaggle, open Notebook Settings, select a GPU accelerator, enable Internet, then run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='cuda:0',
    trust_remote_code=False,
    attn_implementation='eager'
)
model.eval()
print('DREAM starting model loaded:', MODEL_ID)
print('Parameters (billions):', round(sum(p.numel() for p in model.parameters()) / 1e9, 3))


In [ ]:
def dream_reply(user_text, max_new_tokens=220):
    messages = [
        {'role': 'system', 'content': 'You are DREAM AI, an independent assistant being developed in Kenya. Be helpful, clear, honest, and practical. Do not claim to have performed actions you have not performed.'},
        {'role': 'user', 'content': '/no_think\\n' + user_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = generated[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_prompt = 'Introduce yourself as DREAM AI and explain how you can help a learner study mathematics.'
answer = dream_reply(test_prompt)
print('TEST PROMPT:', test_prompt)
print('DREAM AI RESPONSE:\\n', answer)
if not answer:
    raise RuntimeError('The model loaded but returned an empty response; bootstrap test failed.')
print('\\nBOOTSTRAP STATUS: PASS — pretrained model loaded and generated a response.')


In [ ]:
from datetime import datetime, timezone
manifest = {
    'project': 'DREAM AI',
    'base_model': MODEL_ID,
    'model_license': 'Apache-2.0 (verify model card and include notices when redistributing)',
    'bootstrap_status': 'passed',
    'test_prompt': test_prompt,
    'test_response': answer,
    'gpu': torch.cuda.get_device_name(0),
    'torch_version': torch.__version__,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'note': 'This confirms model loading and inference only. It is not evidence of fine-tuning or autonomous continual learning.'
}
(OUTPUT_DIR / 'bootstrap-manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'bootstrap-test.txt').write_text('PROMPT\\n' + test_prompt + '\\n\\nRESPONSE\\n' + answer + '\\n', encoding='utf-8')
print('Saved verification files to:', OUTPUT_DIR)
print('Files:', [p.name for p in OUTPUT_DIR.iterdir()])
print('Next: save a Kaggle notebook version so the successful run and outputs are retained.')


## What happens after this passes?

This gives DREAM a working pretrained brain for the first test. The next engineering step is to connect the model runtime to the DREAM web app and add a controlled LoRA learning worker. That worker will learn from approved examples, evaluate each candidate against a held-out test set, and keep or reject updates. Kaggle itself is temporary compute, not a permanent inference server; saved model weights/adapters and training checkpoints must be stored outside the session.
